# Задание 6.2. Сигнал на входе и выходе канала
**Зелёный уровень**

Рассмотрим перенос примеси вдоль канала длины 1:
$$u_t+cu_x=0,\qquad x\in[0,1],\quad c=1,\quad u(x,0)=0.$$
На входе задаётся концентрация $u(0,t)=q(t)$, где
$$
q(t)=\begin{cases}
\sin^2(\pi t/\tau),& 0\le t\le \tau,\\
0,&\text{иначе},
\end{cases}
\qquad \tau=0.2.
$$

Выполним расчёт до $t=1.5$ на четырёх сетках с числом Куранта $C=0.8$. Сравним профили и выходные сигналы с точным решением, проверим баланс примеси и оценим, как меняются высота, длительность и время прихода импульса.

## 1. Точное решение

При $c>0$ характеристики направлены слева направо. Сигнал, наблюдаемый в точке $x$ в момент $t$, поступил на вход в момент $t-x/c$. Если это время отрицательно, сигнал ещё не дошёл и канал в этой точке остаётся чистым. Таким образом,
$$u(x,t)=q(t-x/c),\qquad u(1,t)=q(t-1).$$

Точный выход — копия входного импульса, задержанная на единицу времени. Она ненулевая между $1$ и $1.2$, её максимум равен $1$ при $t=1.1$.

В численной схеме хранятся **средние по ячейкам**. Для вычисления ошибки будем сравнивать их с точными средними, а не со значениями в центрах. Обозначим $Q(s)=\int_{-\infty}^{s}q(r)\,dr$. Тогда точное среднее по ячейке $[a,b]$:
$$
\bar u_{[a,b]}(t)=\frac{c}{b-a}
\left[Q(t-a/c)-Q(t-b/c)\right].
$$
Первообразная вычисляется аналитически: если $r=\min(\tau,\max(0,s))$, то
$$Q(s)=\frac{r}{2}-\frac{\tau}{4\pi}\sin\frac{2\pi r}{\tau}.$$
За весь импульс в канал входит $cQ(\tau)=0.1$ единицы количества примеси.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

c, tau, t_end, C = 1.0, 0.2, 1.5, 0.8
counts = [80, 160, 320, 640]
colors = ["#b86743", "#28728a", "#458353", "#644c91"]
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18,
    "animation.frame_format": "jpeg", "animation.embed_limit": 30,
})


def q(t):
    t = np.asarray(t)
    return np.where((t >= 0) & (t <= tau), np.sin(np.pi*t/tau)**2, 0.0)


def primitive(t):
    r = np.clip(t, 0, tau)
    return r/2 - tau/(4*np.pi)*np.sin(2*np.pi*r/tau)


def exact_average(faces, t):
    return -c*np.diff(primitive(t-faces/c))/np.diff(faces)

## 2. Схема upwind и дискретный баланс

Делим канал на $N$ ячеек длины $h=1/N$. Обозначим через $U_j^n$ среднее в ячейке $j$ при $t_n=n\Delta t$, а через $F_j^n$ — поток через грань $x=jh$. Ячейки нумеруются от $0$ до $N-1$, грани — от $0$ до $N$.

Поскольку $c>0$, на внутренней грани поток берёт концентрацию из ячейки слева. Вход и выход задаются так:
$$
F_0^n=cq(t_n),\qquad
F_j^n=cU_{j-1}^n,\quad 1\le j\le N.
$$
В частности, $F_N^n=cU_{N-1}^n$ — поток из последней ячейки. **Дополнительного условия на выходе нет.** Обновление ячейки:
$$
U_j^{n+1}=U_j^n-\frac{\Delta t}{h}(F_{j+1}^n-F_j^n).
$$
Для внутренних ячеек это взвешенное среднее $(1-C)U_j^n+C U_{j-1}^n$. При $0\le C\le1$ веса неотрицательны. Во всех опытах берём $\Delta t=0.8h/c$; выбранные шаги делят $t=1.5$ без остатка.

Если просуммировать обновление по всем ячейкам, внутренние потоки сократятся. Останется
$$
M^{n+1}-M^n=\Delta t(F_0^n-F_N^n),\qquad
M^n=h\sum_{j=0}^{N-1}U_j^n.
$$
Накопленные приток $I^n$ и отток $O^n$ считаются по тем же потокам и тем же шагам, что и обновление $U$. Невязка:
$$R^n=M^n-M^0-I^n+O^n.$$
В точной арифметике она равна нулю. Этот баланс проверяет количество примеси, а форму профиля проверяем отдельно.

In [ ]:
def solve_channel(N):
    h = 1.0/N
    dt = C*h/c
    steps = int(round(t_end/dt))
    if not np.isclose(steps*dt, t_end, atol=1e-12, rtol=0):
        raise ValueError("Выберите сетку, на которой t_end кратно Δt.")

    t = dt*np.arange(steps+1)
    faces = np.linspace(0.0, 1.0, N+1)
    u = np.zeros((steps+1, N))
    flux = np.empty(N+1)
    inlet = q(t)
    fin, fout = np.empty(steps), np.empty(steps)

    for n in range(steps):
        flux[0] = c*inlet[n]
        flux[1:] = c*u[n]
        fin[n], fout[n] = flux[0], flux[-1]
        u[n+1] = u[n] - (dt/h)*np.diff(flux)

    mass = h*u.sum(axis=1)
    incoming = np.r_[0.0, np.cumsum(dt*fin)]
    outgoing = np.r_[0.0, np.cumsum(dt*fout)]
    residual = mass-mass[0]-incoming+outgoing

    # Точки на выходе относятся к началу шага t_n.
    # Последнее значение нужно только для графика при t_end.
    outlet = np.r_[fout/c, u[-1, -1]]
    return dict(h=h, dt=dt, t=t, faces=faces, u=u, outlet=outlet,
                mass=mass, incoming=incoming, outgoing=outgoing,
                residual=residual)


runs = {N: solve_channel(N) for N in counts}
print(f"{'N':>6} {'h':>12} {'Δt':>12} {'C':>8} {'Шагов':>8}")
for N, r in runs.items():
    print(f"{N:6d} {r['h']:12.7f} {r['dt']:12.7f} "
          f"{c*r['dt']/r['h']:8.3f} {len(r['t'])-1:8d}")

## 3. Профиль импульса внутри канала

Сравниваем расчёты в одинаковые моменты физического времени. Точный профиль — непрерывная кривая, численный — ступеньки со значениями средних в ячейках. К $t=1.1$ часть импульса уже вышла, а к $t=1.24$ точный профиль равен нулю во всём канале.

Для численного сравнения при $t=0.6$ измеряем среднюю интегральную ошибку по каналу:
$$E_x=h\sum_j |U_j-\bar u_j^{\mathrm{exact}}|.$$
Здесь $\bar u_j^{\mathrm{exact}}$ — аналитические средние из раздела 1. Длина канала равна единице, поэтому $E_x$ также равна средней абсолютной ошибке по пространству.

In [ ]:
def time_index(r, t):
    n = int(round(t/r["dt"]))
    if not np.isclose(r["t"][n], t, atol=1e-12, rtol=0):
        raise ValueError("Момент сравнения не попал на временную сетку.")
    return n


sample_times = [0.30, 0.60, 1.10, 1.24]
xx = np.linspace(0, 1, 1001)
fig, axes = plt.subplots(2, 2, figsize=(10.5, 6.2), sharex=True, sharey=True,
                         constrained_layout=True)
for ax, t in zip(axes.flat, sample_times):
    for N, color in zip(counts, colors):
        r = runs[N]
        ax.stairs(r["u"][time_index(r, t)], r["faces"], baseline=None,
                  lw=1.1, color=color, label=f"N = {N}")
    ax.plot(xx, q(t-xx/c), "k--", lw=1.45, label="Точное")
    ax.set(title=f"t = {t:.2f}", xlim=(0, 1), ylim=(-0.04, 1.08),
           xlabel="x", ylabel="Концентрация")
axes[0, 0].legend(fontsize=8, ncol=2, loc="upper right")
fig.suptitle("Перенос импульса: сравнение на общих моментах времени")
plt.show()

profile_error = {}
for N, r in runs.items():
    t = 0.6
    j = time_index(r, t)
    profile_error[N] = r["h"]*np.abs(r["u"][j]-exact_average(r["faces"], t)).sum()

## 4. Входной и выходной сигналы

Выходная концентрация в численном расчёте — $F_N^n/c=U_{N-1}^n$ при $t=t_n$. Сопоставим её с $q(t-1)$ и отдельно покажем исходный сигнал $q(t)$.

Измеряем четыре характеристики:

- **Высота** — максимальное значение на временной сетке.
- **Момент пика** — время этого узлового максимума; его точность ограничена шагом времени.
- **Длительность** — ширина на половине *собственного* максимума (FWHM).
- **Момент прихода** — первое пересечение общего порога $0.05$ снизу вверх. Порог позволяет не принимать за приход практически нулевой хвост размытого решения.

Время пересечения уровня уточняем линейно между соседними отсчётами. Для точного выхода ширина на полувысоте равна $0.1$, а момент прихода по порогу $0.05$ равен $1+\frac{\tau}{\pi}\arcsin\sqrt{0.05}$. Этот момент отличается от начала точного ненулевого сигнала ($t=1$), потому что после $t=1$ ещё нужно набрать высоту $0.05$.

In [ ]:
def crossing(t, signal, level, j):
    # Время пересечения между узлами j и j+1.
    fraction = (level-signal[j])/(signal[j+1]-signal[j])
    return t[j]+fraction*(t[j+1]-t[j])


def signal_metrics(t, signal, threshold=0.05):
    peak_index = int(np.argmax(signal))
    peak = signal[peak_index]
    half = np.flatnonzero(signal >= peak/2)
    first = np.flatnonzero((signal[:-1] < threshold) & (signal[1:] >= threshold))[0]
    left = crossing(t, signal, peak/2, half[0]-1)
    right = crossing(t, signal, peak/2, half[-1])
    return peak, t[peak_index], right-left, crossing(t, signal, threshold, first)


threshold = 0.05
exact_arrival = 1/c+tau/np.pi*np.arcsin(np.sqrt(threshold))
metrics = {N: signal_metrics(r["t"], r["outlet"], threshold) for N, r in runs.items()}
print(f"{'Сетка':>8} {'Высота':>11} {'Время пика':>13} {'Ширина':>11} {'Приход':>11}")
for N, values in metrics.items():
    print(f"{N:8d} {values[0]:11.6f} {values[1]:13.6f} "
          f"{values[2]:11.6f} {values[3]:11.6f}")
print(f"{'Точно':>8} {1.0:11.6f} {1/c+tau/2:13.6f} "
      f"{tau/2:11.6f} {exact_arrival:11.6f}")

t_dense = np.linspace(0, t_end, 1501)
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.3),
                         gridspec_kw={"width_ratios": [1.45, 1]},
                         constrained_layout=True)
axes[0].plot(t_dense, q(t_dense), color="0.55", lw=1.6, label="Вход")
for ax in axes:
    for (N, r), color in zip(runs.items(), colors):
        ax.plot(r["t"], r["outlet"], color=color, lw=1.35, label=f"Выход, N = {N}")
    ax.plot(t_dense, q(t_dense-1/c), "k--", lw=1.45, label="Точный выход")
    ax.set(xlabel="t", ylabel="Концентрация")
axes[0].set(title="Сигналы на входе и выходе", xlim=(0, 1.5), ylim=(-0.02, 1.06))
axes[0].legend(fontsize=8, loc="upper center", ncol=2)
axes[1].axhline(threshold, color="0.55", ls=":", lw=1)
axes[1].scatter([values[3] for values in metrics.values()], [threshold]*len(counts),
                c=colors, s=22, zorder=5)
axes[1].scatter(exact_arrival, threshold, color="black", s=22, zorder=5)
axes[1].set(title="Передний край: пересечение уровня 0.05",
            xlim=(0.935, 1.06), ylim=(-0.006, 0.4))
plt.show()

На грубой сетке $N=80$ пик падает с $1$ до $0.641$, а ширина на полувысоте увеличивается с $0.100$ до $0.147$. При сгущении сетки выход становится выше и уже: при $N=640$ высота $0.929$, ширина $0.105$.

Передний край размывается: уровень $0.05$ достигается раньше точного момента $1.01436$. Например, при $N=80$ это происходит в $0.96359$, а при $N=640$ — в $1.00414$. Раннее достижение малого порога и положение максимума — разные характеристики. На всех четырёх сетках наибольший отсчёт сигнала находится при $t=1.10$; заметного запаздывания узлового пика нет.

Обновление upwind смешивает соседние значения, поэтому оно сглаживает импульс. Ведущий коэффициент численной диффузии (см. конспект занятия) равен $D_{\mathrm{num}}=ch(1-C)/2$. При фиксированном $C$ он уменьшается вместе с $h$, что объясняет ослабление размытия на мелких сетках.

In [ ]:
def trapezoid_integral(values, t):
    # Площадь каждой трапеции: среднее соседних высот × длина шага.
    return np.sum(0.5*(values[:-1]+values[1:])*np.diff(t))


outlet_error = {
    N: trapezoid_integral(np.abs(r["outlet"]-q(r["t"]-1/c)), r["t"])
    for N, r in runs.items()
}
print("Ошибки: E_x — профиль при t=0.6, E_t — интеграл ошибки выхода по времени.")
print(f"{'N':>6} {'E_x':>13} {'E_t':>13}")
for N in counts:
    print(f"{N:6d} {profile_error[N]:13.6e} {outlet_error[N]:13.6e}")

h_values = np.array([runs[N]["h"] for N in counts])
fig, ax = plt.subplots(figsize=(7.5, 4.1), constrained_layout=True)
ax.loglog(h_values, [profile_error[N] for N in counts], "o-", color=colors[1],
          label="Ошибка профиля при t = 0.6")
ax.loglog(h_values, [outlet_error[N] for N in counts], "s-", color=colors[0],
          label="Ошибка выходного сигнала")
ax.loglog(h_values, profile_error[640]*h_values/h_values[-1], "k:",
          label="Прямая наклона 1")
ax.set(xlabel="Размер ячейки h", ylabel="Интегральная абсолютная ошибка",
       title="Точность при сгущении сетки, C = 0.8")
ax.legend(fontsize=9)
plt.show()

## 5. Баланс количества примеси

На первом графике показаны $M^n$, $I^n$ и $O^n$ для сетки $N=160$. Пока примесь поступает, её количество в канале растёт. После окончания подачи и до прихода к выходу оно почти постоянно; затем уменьшается, а накопленный отток растёт. К концу расчёта практически вся примесь вышла.

На втором графике — невязка $R^n$ для всех сеток. Значения порядка $10^{-16}$ соответствуют округлению. Использование аналитического выходного сигнала вместо $F_N^n$ в накопленном оттоке нарушило бы эту проверку, поскольку это был бы другой поток.

In [ ]:
demo = runs[160]
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.1), constrained_layout=True)
axes[0].plot(demo["t"], demo["mass"], lw=2, color=colors[1], label="В канале M")
axes[0].plot(demo["t"], demo["incoming"], "--", lw=1.5, color="0.45",
             label="Накопленный приток I")
axes[0].plot(demo["t"], demo["outgoing"], lw=1.7, color=colors[0],
             label="Накопленный отток O")
axes[0].set(title="Количество примеси, N = 160", ylabel="Количество")
axes[0].legend(fontsize=9)
for (N, r), color in zip(runs.items(), colors):
    axes[1].plot(r["t"], 1e16*r["residual"], lw=0.9, color=color, label=f"N = {N}")
axes[1].set(title="Невязка дискретного баланса", ylabel=r"$R^n / 10^{-16}$")
axes[1].legend(fontsize=8, ncol=2)
for ax in axes:
    ax.set(xlabel="t", xlim=(0, 1.5))
plt.show()

print(f"{'N':>6} {'Приток к 1.5':>15} {'Отток к 1.5':>15} "
      f"{'Остаток к 1.5':>15} {'max |R|':>13}")
for N, r in runs.items():
    print(f"{N:6d} {r['incoming'][-1]:15.10f} {r['outgoing'][-1]:15.10f} "
          f"{r['mass'][-1]:15.3e} {np.max(np.abs(r['residual'])):13.3e}")

## 6. Анимация прохождения импульса

Слева — профиль в канале ($N=160$) и точное решение. Справа — выходные сигналы; вертикальная линия указывает то же время, что и профиль слева. В кадры попадает каждый третий шаг интегрирования, но расчёт выполняется на всех шагах. Нажмите ▶ для воспроизведения.

In [ ]:
frame_nodes = np.arange(0, len(demo["t"]), 3)
fig, (ax_u, ax_out) = plt.subplots(1, 2, figsize=(10.4, 4.0))
fig.subplots_adjust(left=0.075, right=0.98, bottom=0.15, top=0.85, wspace=0.27)

profile = ax_u.stairs(demo["u"][0], demo["faces"], baseline=None,
                      color=colors[1], lw=1.8, label="Upwind")
exact_line, = ax_u.plot(xx, q(-xx/c), "k--", lw=1.4, label="Точное")
ax_u.set(xlabel="x", ylabel="Концентрация", xlim=(0, 1), ylim=(-0.03, 1.06),
         title="Профиль в канале")
ax_u.legend(loc="upper right", fontsize=8)

ax_out.plot(demo["t"], demo["outlet"], color=colors[1], lw=1.6, label="Upwind")
ax_out.plot(t_dense, q(t_dense-1/c), "k--", lw=1.4, label="Точный выход")
cursor = ax_out.axvline(0, color=colors[0], lw=1.3)
point, = ax_out.plot([0], [0], "o", color=colors[1], ms=5)
ax_out.set(xlabel="t", ylabel="Концентрация", xlim=(0, 1.5), ylim=(-0.03, 1.06),
           title="Сигнал на выходе")
ax_out.legend(loc="upper left", fontsize=8)
title = fig.suptitle("")


def update(n):
    t = demo["t"][n]
    profile.set_data(values=demo["u"][n])
    exact_line.set_ydata(q(t-xx/c))
    cursor.set_xdata([t, t])
    point.set_data([t], [demo["outlet"][n]])
    title.set_text(f"Прохождение импульса · t = {t:.3f}")
    return profile, exact_line, cursor, point, title


movie = FuncAnimation(fig, update, frames=frame_nodes, interval=80, blit=False)
with plt.rc_context({"savefig.dpi": 75}):
    animation = movie.to_jshtml(fps=12, default_mode="once")
plt.close(fig)
display(HTML(animation))

## Выводы

1. При $c=1$ точный выход повторяет вход с задержкой 1. Схема upwind даёт более низкий и широкий импульс: на сетке $N=80$ пик ниже точного примерно на 36%, а ширина на полувысоте больше примерно на 47%.
2. Слабая передняя часть численного сигнала приходит раньше: фиксированный уровень $0.05$ пересекается раньше точного. При этом узловой максимум в этих опытах приходится на $t=1.10$. Ранний фронт не означает такого же сдвига всего импульса.
3. При увеличении $N$ от 80 до 640 при одном и том же $C=0.8$ уменьшаются ошибки профиля и выходного сигнала, восстанавливается высота и уменьшается ширина.
4. Баланс притока, оттока и количества примеси внутри канала выполняется с точностью округления на всех сетках. Сохранение количества примеси совместимо с заметным искажением её распределения.